# 02 - Model Training & Evaluation

## Insurance Claims Prediction ML Project

### Overview
This notebook trains and evaluates four classification models for predicting insurance claims:

1. **Logistic Regression** - Linear baseline with L2 regularization
2. **Random Forest** - Ensemble of decision trees with bagging
3. **XGBoost** - Gradient boosted trees with regularization
4. **LightGBM** - Fast gradient boosting with histogram-based splitting

### Key Techniques
- **Stratified K-Fold Cross-Validation** for robust evaluation
- **Probability Calibration** (Platt scaling & isotonic regression) for reliable risk scores
- **Threshold Optimization** (F1, Youden's J, business value) for optimal decision boundaries
- **SHAP Explainability** for global and local feature importance

### Notebook Outline
1. Imports and setup
2. Data loading and preparation
3. Train/test split
4. Model training with cross-validation
5. Evaluation metrics and confusion matrices
6. ROC curve comparison
7. Probability calibration
8. Threshold optimization
9. SHAP analysis
10. Model comparison summary
11. Conclusion

## 1. Imports and Setup

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, brier_score_loss,
    precision_recall_curve, f1_score, log_loss, make_scorer
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Gradient boosting
import xgboost as xgb
import lightgbm as lgb

# Explainability
import shap

# Add project root to path for src imports
sys.path.insert(0, '..')
from src.model_training import get_models, cross_validate_models
from src.calibration import calibrate_model, plot_calibration_curve
from src.threshold_optimizer import find_optimal_threshold, plot_threshold_analysis
from src.explainability import explain_global, explain_local

# Plot styling
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('whitegrid')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries loaded successfully.")
print(f"  XGBoost:  {xgb.__version__}")
print(f"  LightGBM: {lgb.__version__}")
print(f"  SHAP:     {shap.__version__}")

## 2. Load and Prepare Data

Same synthetic generation approach as Notebook 01 - ensures this notebook is self-contained.

In [ ]:
def generate_synthetic_insurance_data(n_samples=10000, random_state=42):
    """
    Generate a synthetic insurance claims dataset with realistic distributions.
    Used as a fallback when the real Kaggle dataset is not available.
    """
    rng = np.random.RandomState(random_state)
    
    age = rng.normal(loc=40, scale=12, size=n_samples).clip(18, 75).astype(int)
    gender = rng.choice(['Male', 'Female'], size=n_samples, p=[0.55, 0.45])
    vehicle_age = rng.choice(
        ['< 1 Year', '1-2 Year', '> 2 Years'],
        size=n_samples, p=[0.25, 0.40, 0.35]
    )
    vehicle_type = rng.choice(
        ['Sedan', 'SUV', 'Hatchback', 'Truck', 'Van'],
        size=n_samples, p=[0.35, 0.25, 0.20, 0.12, 0.08]
    )
    annual_premium = rng.lognormal(mean=9.5, sigma=0.6, size=n_samples).clip(5000, 80000).astype(int)
    policy_tenure = rng.exponential(scale=5, size=n_samples).clip(0.5, 30).round(1)
    num_claims_hist = rng.poisson(lam=0.8, size=n_samples)
    credit_score = rng.normal(loc=680, scale=80, size=n_samples).clip(300, 850).astype(int)
    region = rng.choice(['North', 'South', 'East', 'West'], size=n_samples, p=[0.30, 0.25, 0.20, 0.25])
    
    # Target with realistic correlations
    claim_prob = (
        0.05
        + 0.15 * (age < 30).astype(float)
        + 0.10 * (vehicle_age == '> 2 Years').astype(float)
        + 0.08 * (num_claims_hist >= 2).astype(float)
        + 0.10 * (credit_score < 600).astype(float)
        + 0.05 * (annual_premium > 25000).astype(float)
        + rng.normal(0, 0.05, n_samples)
    ).clip(0.01, 0.95)
    claim_filed = rng.binomial(1, claim_prob)
    
    return pd.DataFrame({
        'age': age, 'gender': gender, 'vehicle_age': vehicle_age,
        'annual_premium': annual_premium, 'policy_tenure': policy_tenure,
        'num_claims_hist': num_claims_hist, 'credit_score': credit_score,
        'region': region, 'vehicle_type': vehicle_type, 'claim_filed': claim_filed
    })


# Load or generate data
data_dir = os.path.join('..', 'data')
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')] if os.path.exists(data_dir) else []

if csv_files:
    df = pd.read_csv(os.path.join(data_dir, csv_files[0]))
    print(f"Loaded real dataset: {df.shape}")
else:
    df = generate_synthetic_insurance_data(n_samples=10000)
    print(f"Generated synthetic dataset: {df.shape}")

print(f"Columns: {df.columns.tolist()}")
print(f"Target distribution:\n{df['claim_filed'].value_counts()}")

In [ ]:
# Feature engineering (same as Notebook 01)
target_col = 'claim_filed'

# Interaction features
if 'annual_premium' in df.columns and 'age' in df.columns:
    df['premium_per_age'] = df['annual_premium'] / (df['age'] + 1)

if 'num_claims_hist' in df.columns and 'policy_tenure' in df.columns:
    df['claims_per_tenure'] = df['num_claims_hist'] / (df['policy_tenure'] + 0.1)

if 'annual_premium' in df.columns and 'credit_score' in df.columns:
    df['premium_credit_ratio'] = df['annual_premium'] / (df['credit_score'] + 1)

# Log transform
if 'annual_premium' in df.columns:
    df['log_premium'] = np.log1p(df['annual_premium'])

# Vehicle age numeric
if 'vehicle_age' in df.columns and df['vehicle_age'].dtype == 'object':
    df['vehicle_age_numeric'] = df['vehicle_age'].map(
        {'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2}
    ).fillna(1)

print(f"Features after engineering: {df.shape[1]}")
print(f"Engineered columns: {df.columns.tolist()}")

## 3. Train/Test Split with Stratification

We encode categorical features, scale numerics, and split the data using stratification to preserve the class distribution.

In [ ]:
# Separate target
y = df[target_col].values
X = df.drop(columns=[target_col])

# Encode categorical variables
label_encoders = {}
for col in X.select_dtypes(include=['object', 'category']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"  Encoded: {col} ({len(le.classes_)} classes)")

feature_names = X.columns.tolist()
print(f"\nTotal features: {len(feature_names)}")
print(f"Features: {feature_names}")

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set:  {X_test.shape[0]} samples")
print(f"Positive rate - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")

## 4. Train Four Models with Cross-Validation

We train all four models using 5-fold stratified cross-validation to get robust performance estimates before evaluating on the held-out test set.

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(
        C=1.0, penalty='l2', solver='lbfgs', max_iter=1000,
        class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_split=5,
        min_samples_leaf=2, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=1,
        eval_metric='logloss', random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=0
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, is_unbalance=True,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
}

print(f"Models to train: {list(models.keys())}")

In [ ]:
# Cross-validation with multiple metrics
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'neg_brier': make_scorer(brier_score_loss, response_method='predict_proba', greater_is_better=False),
    'neg_log_loss': 'neg_log_loss',
}

cv_results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Cross-validating: {name}")
    print(f"{'='*60}")
    
    scores = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring=scoring, return_train_score=False, n_jobs=-1
    )
    
    cv_results[name] = {
        'accuracy': scores['test_accuracy'].mean(),
        'accuracy_std': scores['test_accuracy'].std(),
        'roc_auc': scores['test_roc_auc'].mean(),
        'roc_auc_std': scores['test_roc_auc'].std(),
        'brier_score': -scores['test_neg_brier'].mean(),
        'log_loss': -scores['test_neg_log_loss'].mean(),
    }
    
    print(f"  Accuracy:    {cv_results[name]['accuracy']:.4f} (+/- {cv_results[name]['accuracy_std']:.4f})")
    print(f"  AUC-ROC:     {cv_results[name]['roc_auc']:.4f} (+/- {cv_results[name]['roc_auc_std']:.4f})")
    print(f"  Brier Score: {cv_results[name]['brier_score']:.4f}")
    print(f"  Log Loss:    {cv_results[name]['log_loss']:.4f}")

# Display CV results as a table
cv_df = pd.DataFrame(cv_results).T
print("\n" + "="*60)
print("Cross-Validation Summary")
print("="*60)
cv_df.round(4)

## 5. Evaluation on Test Set

Train each model on the full training set and evaluate on the held-out test set.

In [ ]:
# Train all models on full training set and evaluate on test set
trained_models = {}
test_results = {}
y_probs = {}  # Store predicted probabilities for later use

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training & Evaluating: {name}")
    print(f"{'='*60}")
    
    # Train
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_probs[name] = y_prob
    
    # Compute metrics
    test_results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'brier_score': brier_score_loss(y_test, y_prob),
        'log_loss': log_loss(y_test, y_prob),
        'f1': f1_score(y_test, y_pred),
    }
    
    print(f"  Accuracy:    {test_results[name]['accuracy']:.4f}")
    print(f"  AUC-ROC:     {test_results[name]['roc_auc']:.4f}")
    print(f"  Brier Score: {test_results[name]['brier_score']:.4f}")
    print(f"  F1 Score:    {test_results[name]['f1']:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=['No Claim', 'Claim']))

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
        xticklabels=['No Claim', 'Claim'],
        yticklabels=['No Claim', 'Claim'],
        cbar=False
    )
    axes[i].set_title(f'{name}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices (Test Set)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. ROC Curve Comparison

Comparing the Receiver Operating Characteristic (ROC) curves for all four models on the test set.

In [ ]:
# ROC curves for all models
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

colors = {'Logistic Regression': '#3498db', 'Random Forest': '#e74c3c',
          'XGBoost': '#2ecc71', 'LightGBM': '#f39c12'}

for name in trained_models:
    y_prob = y_probs[name]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    
    # Full ROC curve
    ax1.plot(fpr, tpr, color=colors[name], linewidth=2.5,
             label=f'{name} (AUC = {auc:.4f})')
    
    # Zoomed into top-left corner
    ax2.plot(fpr, tpr, color=colors[name], linewidth=2.5,
             label=f'{name} (AUC = {auc:.4f})')

# Full ROC
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.7, label='Random (AUC = 0.5)')
ax1.set_xlabel('False Positive Rate', fontsize=13)
ax1.set_ylabel('True Positive Rate', fontsize=13)
ax1.set_title('ROC Curve Comparison', fontsize=15, fontweight='bold')
ax1.legend(loc='lower right', fontsize=11)
ax1.grid(True, alpha=0.3)

# Zoomed ROC (top-left corner)
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.7)
ax2.set_xlim([-0.01, 0.3])
ax2.set_ylim([0.7, 1.01])
ax2.set_xlabel('False Positive Rate', fontsize=13)
ax2.set_ylabel('True Positive Rate', fontsize=13)
ax2.set_title('ROC Curve (Zoomed)', fontsize=15, fontweight='bold')
ax2.legend(loc='lower right', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Probability Calibration

Raw predicted probabilities from classifiers may not be well-calibrated (i.e., a predicted probability of 0.3 doesn't necessarily mean a 30% chance of a claim). We apply:

- **Platt Scaling** (sigmoid): Fits a logistic regression on the model outputs
- **Isotonic Regression**: Non-parametric calibration method

We use the best-performing model (by AUC-ROC) for calibration analysis.

In [ ]:
# Select the best model by AUC-ROC for calibration
best_model_name = max(test_results, key=lambda x: test_results[x]['roc_auc'])
best_model = trained_models[best_model_name]
print(f"Best model by AUC-ROC: {best_model_name} ({test_results[best_model_name]['roc_auc']:.4f})")

# Uncalibrated probabilities
y_prob_uncalibrated = y_probs[best_model_name]
brier_uncalibrated = brier_score_loss(y_test, y_prob_uncalibrated)

# Platt scaling (sigmoid)
cal_sigmoid = CalibratedClassifierCV(best_model, method='sigmoid', cv=5)
cal_sigmoid.fit(X_train, y_train)
y_prob_platt = cal_sigmoid.predict_proba(X_test)[:, 1]
brier_platt = brier_score_loss(y_test, y_prob_platt)

# Isotonic regression
cal_isotonic = CalibratedClassifierCV(best_model, method='isotonic', cv=5)
cal_isotonic.fit(X_train, y_train)
y_prob_isotonic = cal_isotonic.predict_proba(X_test)[:, 1]
brier_isotonic = brier_score_loss(y_test, y_prob_isotonic)

print(f"\nBrier Score (lower is better):")
print(f"  Uncalibrated:        {brier_uncalibrated:.4f}")
print(f"  Platt Scaling:       {brier_platt:.4f}")
print(f"  Isotonic Regression: {brier_isotonic:.4f}")

# Select best calibration
if brier_platt <= brier_isotonic:
    best_calibrated = cal_sigmoid
    best_cal_name = 'Platt Scaling'
    best_brier = brier_platt
    y_prob_calibrated = y_prob_platt
else:
    best_calibrated = cal_isotonic
    best_cal_name = 'Isotonic Regression'
    best_brier = brier_isotonic
    y_prob_calibrated = y_prob_isotonic

improvement = (brier_uncalibrated - best_brier) / brier_uncalibrated * 100
print(f"\nBest calibration: {best_cal_name} ({improvement:.1f}% Brier improvement)")

In [ ]:
# Calibration curves (reliability diagrams) - before and after calibration
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

prob_dict = {
    'Uncalibrated': y_prob_uncalibrated,
    'Platt Scaling': y_prob_platt,
    'Isotonic': y_prob_isotonic,
}
colors_cal = ['#3498db', '#e74c3c', '#2ecc71']

# Reliability diagram
for (name, y_prob), color in zip(prob_dict.items(), colors_cal):
    fraction_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=10, strategy='uniform')
    brier = brier_score_loss(y_test, y_prob)
    ax1.plot(mean_pred, fraction_pos, marker='o', linewidth=2, color=color,
             label=f'{name} (Brier={brier:.4f})')
    
    # Probability distribution
    ax2.hist(y_prob, bins=50, alpha=0.4, color=color, label=name, edgecolor='black')

# Perfect calibration line
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect Calibration')
ax1.set_xlabel('Mean Predicted Probability', fontsize=13)
ax1.set_ylabel('Fraction of Positives', fontsize=13)
ax1.set_title('Calibration Curve (Reliability Diagram)', fontsize=15, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Predicted Probability', fontsize=13)
ax2.set_ylabel('Count', fontsize=13)
ax2.set_title('Distribution of Predicted Probabilities', fontsize=15, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Threshold Optimization

The default threshold of 0.5 is often suboptimal, especially for imbalanced datasets. We explore:

- **F1-optimal threshold**: Maximizes the harmonic mean of precision and recall
- **Youden's J statistic**: Maximizes sensitivity + specificity - 1
- **Business value threshold**: Incorporates cost-benefit analysis (catching claims early vs. false investigations)

In [ ]:
# Use calibrated probabilities for threshold optimization
y_prob_opt = y_prob_calibrated

# Compute metrics across all thresholds
thresholds = np.arange(0.01, 1.0, 0.01)
metrics_by_threshold = []

for t in thresholds:
    y_pred_t = (y_prob_opt >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    metrics_by_threshold.append({
        'threshold': t, 'sensitivity': sensitivity, 'specificity': specificity,
        'precision': precision, 'f1': f1, 'accuracy': accuracy
    })

thresh_df = pd.DataFrame(metrics_by_threshold)

# Find optimal thresholds
f1_opt_idx = thresh_df['f1'].idxmax()
f1_opt_threshold = thresh_df.loc[f1_opt_idx, 'threshold']

youden_j = thresh_df['sensitivity'] + thresh_df['specificity'] - 1
youden_opt_idx = youden_j.idxmax()
youden_opt_threshold = thresh_df.loc[youden_opt_idx, 'threshold']

print(f"Optimal Thresholds:")
print(f"  F1-Optimal:    {f1_opt_threshold:.2f} (F1={thresh_df.loc[f1_opt_idx, 'f1']:.4f})")
print(f"  Youden's J:    {youden_opt_threshold:.2f} (J={youden_j[youden_opt_idx]:.4f})")
print(f"  Default (0.5): F1={thresh_df.loc[thresh_df['threshold'] == 0.50, 'f1'].values[0]:.4f}")

In [ ]:
# Threshold analysis plots
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# Plot 1: Metrics vs threshold
axes[0].plot(thresh_df['threshold'], thresh_df['sensitivity'], 'b-', linewidth=2, label='Sensitivity (Recall)')
axes[0].plot(thresh_df['threshold'], thresh_df['specificity'], 'r-', linewidth=2, label='Specificity')
axes[0].plot(thresh_df['threshold'], thresh_df['precision'], 'g-', linewidth=2, label='Precision')
axes[0].plot(thresh_df['threshold'], thresh_df['f1'], 'm-', linewidth=2.5, label='F1 Score')
axes[0].plot(thresh_df['threshold'], thresh_df['accuracy'], 'k--', linewidth=1, label='Accuracy')
axes[0].axvline(f1_opt_threshold, color='purple', linestyle=':', alpha=0.8, label=f'F1 Optimal ({f1_opt_threshold:.2f})')
axes[0].axvline(0.5, color='gray', linestyle=':', alpha=0.5, label='Default (0.50)')
axes[0].set_xlabel('Threshold', fontsize=13)
axes[0].set_ylabel('Score', fontsize=13)
axes[0].set_title('Metrics vs. Decision Threshold', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9, loc='center right')

# Plot 2: F1 vs threshold
axes[1].plot(thresh_df['threshold'], thresh_df['f1'], 'm-', linewidth=2.5)
axes[1].fill_between(thresh_df['threshold'], thresh_df['f1'], alpha=0.2, color='purple')
axes[1].axvline(f1_opt_threshold, color='red', linestyle='--', linewidth=2,
                label=f'Optimal: {f1_opt_threshold:.2f} (F1={thresh_df.loc[f1_opt_idx, "f1"]:.4f})')
axes[1].axvline(0.5, color='gray', linestyle=':', alpha=0.7, label='Default (0.50)')
axes[1].set_xlabel('Threshold', fontsize=13)
axes[1].set_ylabel('F1 Score', fontsize=13)
axes[1].set_title('F1 Score vs. Threshold', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)

# Plot 3: Precision-Recall tradeoff
axes[2].plot(thresh_df['sensitivity'], thresh_df['precision'], 'b-', linewidth=2)
axes[2].scatter(
    [thresh_df.loc[f1_opt_idx, 'sensitivity']], [thresh_df.loc[f1_opt_idx, 'precision']],
    color='red', s=150, zorder=5, label=f'F1-Optimal (t={f1_opt_threshold:.2f})'
)
axes[2].set_xlabel('Recall (Sensitivity)', fontsize=13)
axes[2].set_ylabel('Precision', fontsize=13)
axes[2].set_title('Precision-Recall Trade-off', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=11)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare performance at different thresholds
print("Performance Comparison at Different Thresholds:")
print(f"{'Threshold':<12} {'Accuracy':<12} {'Sensitivity':<14} {'Specificity':<14} {'Precision':<12} {'F1':<10}")
print("-" * 74)

for t, label in [(0.50, 'Default'), (f1_opt_threshold, 'F1-Optimal'), (youden_opt_threshold, "Youden's J")]:
    row = thresh_df.iloc[(thresh_df['threshold'] - t).abs().argsort().iloc[0]]
    print(f"{t:<12.2f} {row['accuracy']:<12.4f} {row['sensitivity']:<14.4f} {row['specificity']:<14.4f} {row['precision']:<12.4f} {row['f1']:<10.4f}  ({label})")

## 9. SHAP Explainability Analysis

Using SHAP (SHapley Additive exPlanations) to understand model predictions:

- **Global explanations**: Which features are most important across all predictions
- **Local explanations**: How each feature contributed to a specific prediction (waterfall plot)

In [ ]:
# Use a tree-based model for SHAP (TreeExplainer is fastest)
# Pick the best tree model from our trained models
tree_model_names = ['XGBoost', 'LightGBM', 'Random Forest']
shap_model_name = max(
    [n for n in tree_model_names if n in trained_models],
    key=lambda x: test_results[x]['roc_auc']
)
shap_model = trained_models[shap_model_name]
print(f"SHAP analysis using: {shap_model_name}")

# Compute SHAP values
print("Computing SHAP values (may take a moment)...")
explainer = shap.TreeExplainer(shap_model)

# Use a sample for efficiency
X_shap_sample = X_train[:500]
shap_values = explainer.shap_values(X_shap_sample)

# Handle multi-output SHAP values (e.g., Random Forest returns list)
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Class 1 (claim) SHAP values

print(f"SHAP values shape: {shap_values.shape}")
print("SHAP computation complete.")

In [ ]:
# Global SHAP Summary Plot (Beeswarm)
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values,
    X_shap_sample,
    feature_names=feature_names,
    show=False,
    max_display=15
)
plt.title(f'SHAP Feature Importance - {shap_model_name} (Beeswarm)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Global SHAP Bar Plot (Mean |SHAP|)
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values,
    X_shap_sample,
    feature_names=feature_names,
    plot_type='bar',
    show=False,
    max_display=15
)
plt.title(f'Mean |SHAP Value| (Feature Importance) - {shap_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Local explanation: Waterfall plot for a single prediction
# Pick a test sample that was predicted as a claim
test_probs = shap_model.predict_proba(X_test)[:, 1]
claim_idx = np.where(test_probs > 0.5)[0]
sample_idx = claim_idx[0] if len(claim_idx) > 0 else 0

print(f"Explaining prediction for test sample {sample_idx}:")
print(f"  Predicted probability of claim: {test_probs[sample_idx]:.4f}")
print(f"  Actual label: {'Claim' if y_test[sample_idx] == 1 else 'No Claim'}")

# Compute SHAP for the single sample
single_shap = explainer.shap_values(X_test[sample_idx:sample_idx+1])
if isinstance(single_shap, list):
    single_shap = single_shap[1]

# Create Explanation object for waterfall plot
expected_val = explainer.expected_value
if isinstance(expected_val, (list, np.ndarray)):
    expected_val = expected_val[1] if len(expected_val) > 1 else expected_val[0]

explanation = shap.Explanation(
    values=single_shap[0],
    base_values=expected_val,
    data=X_test[sample_idx],
    feature_names=feature_names
)

plt.figure(figsize=(12, 8))
shap.waterfall_plot(explanation, max_display=15, show=False)
plt.title(f'SHAP Waterfall - Sample {sample_idx} (Predicted: {test_probs[sample_idx]:.3f})',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP dependence plots for top 4 features
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_4_indices = np.argsort(mean_abs_shap)[-4:][::-1]
top_4_names = [feature_names[i] for i in top_4_indices]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, idx, fname in zip(axes.flat, top_4_indices, top_4_names):
    scatter = ax.scatter(
        X_shap_sample[:, idx], shap_values[:, idx],
        alpha=0.4, s=12, c=shap_values[:, idx], cmap='coolwarm'
    )
    ax.set_xlabel(fname, fontsize=12)
    ax.set_ylabel(f'SHAP value', fontsize=12)
    ax.set_title(f'Dependence: {fname}', fontsize=13, fontweight='bold')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='SHAP value')

plt.suptitle(f'SHAP Dependence Plots (Top 4 Features) - {shap_model_name}',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Model Comparison Summary

In [ ]:
# Comprehensive model comparison table
comparison_data = []

for name in trained_models:
    comparison_data.append({
        'Model': name,
        'CV Accuracy': f"{cv_results[name]['accuracy']:.4f} (+/- {cv_results[name]['accuracy_std']:.4f})",
        'CV AUC-ROC': f"{cv_results[name]['roc_auc']:.4f} (+/- {cv_results[name]['roc_auc_std']:.4f})",
        'Test Accuracy': f"{test_results[name]['accuracy']:.4f}",
        'Test AUC-ROC': f"{test_results[name]['roc_auc']:.4f}",
        'Test F1': f"{test_results[name]['f1']:.4f}",
        'Brier Score': f"{test_results[name]['brier_score']:.4f}",
        'Log Loss': f"{test_results[name]['log_loss']:.4f}",
    })

comparison_df = pd.DataFrame(comparison_data).set_index('Model')
print("=" * 80)
print("COMPLETE MODEL COMPARISON")
print("=" * 80)
comparison_df

In [ ]:
# Visual comparison of key metrics
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

model_names = list(test_results.keys())
x_pos = np.arange(len(model_names))
bar_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

# AUC-ROC
aucs = [test_results[m]['roc_auc'] for m in model_names]
bars = axes[0].bar(x_pos, aucs, color=bar_colors, edgecolor='black', alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(model_names, rotation=20, ha='right')
axes[0].set_title('AUC-ROC (Higher is Better)', fontsize=13, fontweight='bold')
axes[0].set_ylim([min(aucs) - 0.05, max(aucs) + 0.03])
for bar, val in zip(bars, aucs):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

# F1 Score
f1s = [test_results[m]['f1'] for m in model_names]
bars = axes[1].bar(x_pos, f1s, color=bar_colors, edgecolor='black', alpha=0.85)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(model_names, rotation=20, ha='right')
axes[1].set_title('F1 Score (Higher is Better)', fontsize=13, fontweight='bold')
axes[1].set_ylim([min(f1s) - 0.05, max(f1s) + 0.03])
for bar, val in zip(bars, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

# Brier Score
briers = [test_results[m]['brier_score'] for m in model_names]
bars = axes[2].bar(x_pos, briers, color=bar_colors, edgecolor='black', alpha=0.85)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(model_names, rotation=20, ha='right')
axes[2].set_title('Brier Score (Lower is Better)', fontsize=13, fontweight='bold')
axes[2].set_ylim([0, max(briers) + 0.03])
for bar, val in zip(bars, briers):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

for ax in axes:
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Model Performance Comparison (Test Set)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Conclusion

### Model Performance Summary

We trained and evaluated four classification models for insurance claims prediction:

| Aspect | Finding |
|--------|--------|
| **Best Discriminator** | Gradient boosting models (XGBoost/LightGBM) typically achieve the highest AUC-ROC |
| **Baseline** | Logistic Regression provides a solid interpretable baseline |
| **Ensemble** | Random Forest offers a good balance of performance and robustness |

### Key Takeaways

1. **Probability Calibration Matters**: Raw model probabilities may not reflect true claim rates. Platt scaling or isotonic regression improves Brier scores, which is critical for insurance pricing.

2. **Threshold Selection is Business-Critical**: The default 0.5 threshold is rarely optimal. F1-optimal and Youden's J thresholds provide different trade-offs between catching claims (sensitivity) and avoiding false flags (specificity).

3. **SHAP Explainability**: The most influential features typically include historical claims count, credit score, age, and vehicle age - all consistent with actuarial domain knowledge.

4. **Class Imbalance**: Using `class_weight='balanced'` and proper evaluation metrics (AUC-ROC, F1) rather than raw accuracy is essential.

### Production Recommendations
- Use the calibrated model for probability outputs (risk scoring)
- Apply the F1-optimal or business-value threshold for claim/no-claim decisions
- Monitor model performance and recalibrate periodically
- Use SHAP explanations for regulatory compliance and stakeholder communication

### Reproducibility
All source code is available in the `src/` modules:
- `data_pipeline.py` - Data loading, cleaning, feature engineering
- `model_training.py` - Model training and cross-validation
- `calibration.py` - Probability calibration
- `threshold_optimizer.py` - Threshold optimization
- `explainability.py` - SHAP analysis